# 2) Adapter Egitimi (Colab)

Bu notebook tek crop icin adapter egitir, OOD hazirligini olcer ve export ciktisini kaydeder.

Onerilen kullanim sirasi:
1. Calisma kimligi hucresinde `CROP_NAME` ve `PART_NAME` belirleyin.
2. Parametreleri duzenleyin.
3. Guncelleme/erisim kontrolu hucresini calistirin.
4. Hucreleri sirayla calistirin.
5. Is bitince `guided/00_start_here.md` ile ciktilari okuyun.


In [7]:
from google.colab import userdata

import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit

HF_TOKEN_NAMES = ("HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_HUB_TOKEN")
GITHUB_TOKEN_NAMES = ("GH_TOKEN", "GITHUB_TOKEN")

def _resolve_token(token_names: tuple[str, ...], canonical_env_name: str) -> str | None:
    for env_name in token_names:
        token = str(os.environ.get(env_name, "")).strip()
        if token:
            os.environ.setdefault(canonical_env_name, token)
            return token
    for secret_name in token_names:
        try:
            token = str(userdata.get(secret_name) or "").strip()
        except Exception:
            token = ""
        if token:
            os.environ[canonical_env_name] = token
            return token
    return None

hf_token = _resolve_token(HF_TOKEN_NAMES, "HF_TOKEN")
gh_token = _resolve_token(GITHUB_TOKEN_NAMES, "GH_TOKEN")
if gh_token:
    print('[GIT] GitHub token Colab secret/env uzerinden bulundu.')
else:
    print('[GIT] GitHub token bulunamadi. Public read disinda clone/push gerekiyorsa GH_TOKEN ekleyin.')

if hf_token:
    print('[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.')
else:
    print('[HF] Hugging Face token bulunamadi. Gated model gerekiyorsa Colab secret ekleyin.')

CLONE_TARGET = Path('/content/bitirmeprojesi')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')

def _is_repo_root_inline(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _build_repo_access_url(repo_url: str, token: str | None) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root_inline(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root_inline(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root_inline(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root_inline(child):
                return child
    return None

def _ensure_repo_root_for_update_check() -> Path | None:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, gh_token)
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    return _find_repo_root_inline()

repo_root_for_update_check = _ensure_repo_root_for_update_check()
if repo_root_for_update_check is not None:
    if str(repo_root_for_update_check) not in sys.path:
        sys.path.insert(0, str(repo_root_for_update_check))
    try:
        from scripts.colab_repo_bootstrap import probe_repo_update_status

        update_status = probe_repo_update_status(repo_root_for_update_check)
        relation = str(update_status.get('relation', 'unknown'))
        if relation == 'up_to_date':
            print('[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.')
        elif relation == 'update_available':
            print(f"[KONTROL] Ilk hucre: guncelleme var. Branch={update_status.get('branch', '')}")
        else:
            print(f"[KONTROL] Ilk hucre: guncelleme durumu okunamadi: {update_status.get('message', 'bilgi yok')}")
    except Exception as exc:
        print(f'[KONTROL] Ilk hucrede guncellik kontrolu basarisiz: {exc}')
else:
    print('[KONTROL] Ilk hucrede repo bulunamadi ve auto-clone da basarisiz oldu.')


[GIT] GitHub token Colab secret/env uzerinden bulundu.
[HF] Hugging Face token bulundu. Gerekirse gated model indirmelerinde kullanilacak.
[KONTROL] Ilk hucre: notebook/repo guncel gorunuyor.


In [8]:
# Notebook 2 calisma kimligi
# Once bu hucreyi duzenleyin, sonra bootstrap hucresini yeniden calistirin.
CROP_NAME = "grape"
PART_NAME = "leaf"
ENABLE_BAYESIAN_OPTIMIZATION = False  # Bayesian optimization devre disi
print(f"[RUN] crop={CROP_NAME} part={PART_NAME} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}")


[RUN] crop=grape part=leaf bayes_opt=False


In [9]:
import io
import json
import os
import random
import shutil
import sys
import subprocess
import time
from contextlib import contextmanager
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlsplit, urlunsplit

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch

# --- Bootstrap: repo kokunu bul, bagimliliklari kur ---
GITHUB_TOKEN_NAMES = ('GH_TOKEN', 'GITHUB_TOKEN')
CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/bitirmeprojesi.git')
COMMON_REPO_CANDIDATES = (
    Path('/content/bitirme projesi'),
    CLONE_TARGET,
    Path('/content/aads_ulora'),
)

def _is_repo_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'config').is_dir() and (path / 'scripts').is_dir()

def _running_in_colab_inline() -> bool:
    try:
        import google.colab  # noqa: F401
    except Exception:
        return False
    return True

def _resolve_colab_secret_inline(secret_name: str) -> str:
    if not _running_in_colab_inline():
        return ''
    try:
        from google.colab import userdata
        return str(userdata.get(secret_name) or '').strip()
    except Exception:
        return ''

def _resolve_github_token_inline() -> str:
    for env_name in GITHUB_TOKEN_NAMES:
        token = str(os.environ.get(env_name, '')).strip()
        if token:
            os.environ.setdefault('GH_TOKEN', token)
            return token
    for secret_name in GITHUB_TOKEN_NAMES:
        token = _resolve_colab_secret_inline(secret_name)
        if token:
            os.environ['GH_TOKEN'] = token
            return token
    return ''

def _build_repo_access_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    parsed = urlsplit(str(repo_url or '').strip())
    if parsed.scheme != 'https' or not parsed.netloc:
        return repo_url
    netloc = parsed.netloc.split('@', 1)[-1]
    return urlunsplit((parsed.scheme, f'{token}@{netloc}', parsed.path, parsed.query, parsed.fragment))

def _find_repo_root_inline() -> Path | None:
    for raw in (os.environ.get('AADS_REPO_ROOT'), os.environ.get('REPO_ROOT')):
        if not raw:
            continue
        candidate = Path(raw).expanduser().resolve()
        if _is_repo_root(candidate):
            return candidate
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if _is_repo_root(candidate):
            return candidate
    for candidate in COMMON_REPO_CANDIDATES:
        if _is_repo_root(candidate):
            return candidate
    if CLONE_TARGET.exists() and any(CLONE_TARGET.iterdir()):
        for child in CLONE_TARGET.iterdir():
            if child.is_dir() and _is_repo_root(child):
                return child
    return None

def _ensure_repo_root_for_bootstrap() -> Path:
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    clone_url = _build_repo_access_url(REPO_URL, _resolve_github_token_inline())
    CLONE_TARGET.parent.mkdir(parents=True, exist_ok=True)
    completed = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, str(CLONE_TARGET)],
        check=False,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if completed.stdout:
        print(completed.stdout)
    repo_root = _find_repo_root_inline()
    if repo_root is not None:
        return repo_root
    raise RuntimeError(
        'Repo bootstrap basarisiz oldu. Mevcut bir checkout icin AADS_REPO_ROOT ayarlayin veya '
        'private GitHub auto-clone icin GH_TOKEN/GITHUB_TOKEN saglayin.'
    )

BOOTSTRAP_ROOT = _ensure_repo_root_for_bootstrap()
os.chdir(BOOTSTRAP_ROOT)
if str(BOOTSTRAP_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_ROOT))

from scripts.colab_repo_bootstrap import (
    export_current_colab_notebook,
    install_colab_requirements,
    login_and_check_hf_token,
    mirror_checkpoint_state_to_repo,
    mirror_path_to_repo,
    push_repo_run_to_github,
    resolve_github_token,
    resolve_hf_token,
    resolve_repo_root,
    running_in_colab,
)

IN_COLAB = running_in_colab()

ROOT = resolve_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

install_colab_requirements(ROOT / 'colab_notebooks' / 'requirements_colab.txt', IN_COLAB)

from scripts.colab_checkpointing import TrainingCheckpointManager
from scripts.colab_live_telemetry import ColabLiveTelemetry
from src.core.config_manager import ConfigurationManager
from src.shared.adapter_paths import build_adapter_bundle_root
from scripts.colab_notebook_helpers import build_notebook_run_dir, build_notebook_run_id


def _install_capture_cell_output_compat() -> bool:
    if hasattr(ColabLiveTelemetry, 'capture_cell_output'):
        return False

    def _slugify_capture_name(value: str) -> str:
        slug = ''.join(ch.lower() if ch.isalnum() else '_' for ch in str(value or '').strip())
        while '__' in slug:
            slug = slug.replace('__', '_')
        slug = slug.strip('_')
        return slug or 'cell'

    class _CompatTeeTextStream:
        def __init__(self, *streams):
            self._streams = [stream for stream in streams if stream is not None]

        def write(self, data):
            text = str(data)
            for stream in self._streams:
                stream.write(text)
            return len(text)

        def flush(self):
            for stream in self._streams:
                flush = getattr(stream, 'flush', None)
                if callable(flush):
                    flush()

        def isatty(self):
            return any(bool(getattr(stream, 'isatty', lambda: False)()) for stream in self._streams)

        def writable(self):
            return True

CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED = _install_capture_cell_output_compat()
CROP_NAME = globals().get('CROP_NAME', 'tomato')
PART_NAME = globals().get('PART_NAME', 'unspecified')
RUN_ID = build_notebook_run_id(CROP_NAME, PART_NAME)
NOTEBOOK_FILENAME = '2_interactive_adapter_training.executed.ipynb'
REPO_RUN_DIR = build_notebook_run_dir(ROOT, CROP_NAME, PART_NAME, RUN_ID)
REPO_NOTEBOOK_OUTPUT_PATH = REPO_RUN_DIR / 'notebooks' / NOTEBOOK_FILENAME
LOCAL_OUTPUT_DIR = ROOT / 'outputs' / 'colab_notebook_training'
REPO_OUTPUT_DIR = REPO_RUN_DIR / 'outputs' / 'colab_notebook_training'
REPO_TELEMETRY_DIR = REPO_RUN_DIR / 'telemetry'
REPO_CHECKPOINT_STATE_DIR = REPO_RUN_DIR / 'checkpoint_state'

LOCAL_TELEMETRY_ROOT = ROOT / 'outputs' / 'colab_notebook_training' / 'telemetry_runtime'
LOCAL_TELEMETRY_SPOOL_ROOT = ROOT / '.runtime_tmp' / 'colab_notebook_training' / 'telemetry_spool'

TELEMETRY = ColabLiveTelemetry(
    notebook_name='2_interactive_adapter_training.ipynb',
    run_id=RUN_ID,
    drive_root=LOCAL_TELEMETRY_ROOT,
    local_root=LOCAL_TELEMETRY_SPOOL_ROOT,
)
CHECKPOINT_ROOT = TELEMETRY.drive_run_dir
CHECKPOINT_MANAGER = TrainingCheckpointManager(CHECKPOINT_ROOT, retention=3)
DEVICE = str(ConfigurationManager(config_dir=str(ROOT / 'config'), environment='colab').load_all_configs().get('training', {}).get('continual', {}).get('device', 'cuda'))

LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_NOTEBOOK_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def rt(message: str, *, phase: str = 'notebook') -> None:
    text = str(message)
    print(text)
    TELEMETRY.emit_log(text, phase=phase, level='info')

def save_run_outputs_to_repo() -> dict[str, str]:
    exports: dict[str, str] = {}

    mirrored_outputs = mirror_path_to_repo(LOCAL_OUTPUT_DIR, REPO_OUTPUT_DIR, exclude_dir_names=("checkpoints", "telemetry_runtime"))
    if mirrored_outputs is not None:
        exports['outputs'] = str(mirrored_outputs)

    telemetry_source = TELEMETRY.drive_run_dir
    mirrored_telemetry = mirror_path_to_repo(telemetry_source, REPO_TELEMETRY_DIR)
    if mirrored_telemetry is not None:
        exports['telemetry'] = str(mirrored_telemetry)

    mirrored_checkpoint_state = mirror_checkpoint_state_to_repo(CHECKPOINT_ROOT, REPO_CHECKPOINT_STATE_DIR)
    if mirrored_checkpoint_state is not None:
        exports['checkpoint_state'] = str(mirrored_checkpoint_state)

    return exports

TELEMETRY.configure_repo_output_export(
    notebook_path=REPO_NOTEBOOK_OUTPUT_PATH,
    export_notebook_fn=export_current_colab_notebook,
)
TELEMETRY.update_latest({'phase': 'bootstrap_ready', 'run_id': RUN_ID})
print(f'[BOOTSTRAP] run_id={RUN_ID} crop={CROP_NAME} part={PART_NAME} capture_cell_output_compat={CAPTURE_CELL_OUTPUT_COMPAT_INSTALLED}')


[BOOTSTRAP] run_id=grape_leaf_2026-04-22_11-07-15 crop=grape part=leaf capture_cell_output_compat=False


In [10]:
with TELEMETRY.capture_cell_output("Cell 3: Parameters"):
    # --- Notebook 2 parametreleri ---
    # Bu hucreyi duzenleyin, sonra kalan hucreleri sirayla calistirin.
    # Kosu kimligi icin CROP_NAME/PART_NAME degerlerini ustteki hucreden yonetin.

    # RUNTIME_DATASET_ROOT: Notebook 0'un yazdigi <dataset_key>/continual|val|test|ood yapisini tutan repo-ici root.
    RUNTIME_DATASET_ROOT = "data/prepared_runtime_datasets/grape__leaf"

    # DATASET_NAME: Notebook 0'un urettigi runtime dataset klasor adi. Bos ise notebook kullaniciya sorar.
    DATASET_NAME = ""

    # OOD_ROOT: Gercek OOD klasoru. Bos ise ASK_FOR_OOD_ROOT=True iken notebook yol sorar; Enter varsa runtime ood/ kullanir.
    OOD_ROOT = "data/ood_dataset/grape_leaf_ood_best"
    ASK_FOR_OOD_ROOT = True

    # CROP_NAME ve PART_NAME, kosu adlandirmasi ve metadata icin kullanilir.
    CROP_NAME = globals().get("CROP_NAME", "tomato")
    PART_NAME = globals().get("PART_NAME", "unspecified")
    ENABLE_BAYESIAN_OPTIMIZATION = bool(globals().get("ENABLE_BAYESIAN_OPTIMIZATION", False))

    # FEW_SHOT_RESEARCH_MODE: True olursa 100 image/class production guardrail'i research kosulari icin bypass edilir.
    FEW_SHOT_RESEARCH_MODE = False
    FEW_SHOT_MIN_CLASS_SAMPLES = 1

    # EPOCHS: train split uzerinden kac tam gecis yapilacagi.
    EPOCHS = 24

    # BATCH_SIZE: optimizer adimi basina ornek sayisi. GPU limitine yakin olacak sekilde artirilabilir.
    BATCH_SIZE = 32

    # LEARNING_RATE: adapter/LoRA parametreleri icin optimizer adim buyuklugu.
    LEARNING_RATE = 1e-4

    # LORA_R: LoRA rank degeri. Buyudukce kapasite ve VRAM/islem maliyeti artar.
    LORA_R = 16

    # LORA_ALPHA: LoRA olcekleme katsayisi. Genelde LORA_R degerinin 2x-4x araliginda kullanilir.
    LORA_ALPHA = 16

    # LORA_DROPOUT: LoRA katmanlarina uygulanan dropout. Buyudukce regularizasyon artar.
    LORA_DROPOUT = 0.2

    # OOD_FACTOR: OOD esik hassasiyetini carpansal olarak ayarlar.
    OOD_FACTOR = 3.25
    SURE_SEMANTIC_PERCENTILE = 90.0
    SURE_CONFIDENCE_PERCENTILE = 97.0
    CONFORMAL_ALPHA = 0.05
    CONFORMAL_METHOD = "raps"
    CONFORMAL_RAPS_LAMBDA = 0.2
    CONFORMAL_RAPS_K_REG = 1

    # BER_ENABLED: eski/yeni sinif enerji ayrimi icin deneysel egitim regularizeri.
    BER_ENABLED = False

    # BER_LAMBDA_OLD / BER_LAMBDA_NEW: eski ve yeni sinif kisimlari icin BER ceza agirliklari.
    BER_LAMBDA_OLD = 0.1
    BER_LAMBDA_NEW = 0.1
    BER_WARMUP_STEPS = 50

    # WEIGHT_DECAY: AdamW weight decay degeri.
    WEIGHT_DECAY = 0.02

    # MIXED_PRECISION: {'off', 'auto', 'fp16', 'bf16'} seceneklerinden biri.
    MIXED_PRECISION = "bf16"

    # GRAD_ACCUM_STEPS: gradient accumulation katsayisi.
    GRAD_ACCUM_STEPS = 1

    # MAX_GRAD_NORM: gradient clipping esigi. 0 olursa clipping kapanir.
    MAX_GRAD_NORM = 1.0

    # LABEL_SMOOTHING: CE label smoothing katsayisi.
    LABEL_SMOOTHING = 0.0

    # LOSS_NAME / LOGITNORM_TAU: varsayilan kayip LogitNorm'dur; CE icin loss_name='cross_entropy' secin.
    LOSS_NAME = "logitnorm"
    LOGITNORM_TAU = 1.0

    # Scheduler ayarlari.
    SCHEDULER_NAME = "cosine"
    SCHEDULER_WARMUP_RATIO = 0.1
    SCHEDULER_MIN_LR = 1e-6

    # Early stopping ayarlari.
    EARLY_STOPPING_PATIENCE = 5
    EARLY_STOPPING_MIN_DELTA = 0.0

    # Tekrarlanabilirlik ve runtime ayarlari.
    DETERMINISTIC = False
    TF32_ENABLED = True
    SEED = 42

    # NUM_WORKERS: dataloader worker sayisi. CPU veri yukleme paralelligini belirler.
    NUM_WORKERS = 12

    # PREFETCH: NUM_WORKERS > 0 iken worker basina prefetch katsayisi.
    PREFETCH = 8

    # PIN_MEMORY: host memory sabitleyerek host-to-GPU aktarimini hizlandirir.
    PIN_MEMORY = True

    # USE_CACHE: destekleniyorsa decode edilmis ornekleri RAM'de tutar.
    USE_CACHE = True

    # CACHE_TRAIN_SPLIT: continual/train split'ini de cache'ler. Yuksek RAM'li Colab icin uygundur.
    CACHE_TRAIN_SPLIT = True

    # VALIDATION_EVERY_N_EPOCHS: her N epoch'ta tam validation calistirir; final epoch her zaman dahildir.
    VALIDATION_EVERY_N_EPOCHS = 1

    # CHECKPOINT_EVERY_N_STEPS / CHECKPOINT_ON_EXCEPTION: notebook checkpoint sikligi ayarlari.
    CHECKPOINT_EVERY_N_STEPS = 250
    CHECKPOINT_ON_EXCEPTION = True

    # STDOUT_BATCH_INTERVAL: canli training ilerleme yazdirma araligi.
    STDOUT_BATCH_INTERVAL = 12

    # RESUME_MODE: "fresh" yeni kosu baslatir, "resume" son checkpointten devam eder.
    RESUME_MODE = "fresh"  # "fresh" or "resume"

    # AUTO_DISCONNECT_RUNTIME: tum final exportlar basariliysa Colab runtime'i kapatir.
    AUTO_DISCONNECT_RUNTIME = True

    # AUTO_DISCONNECT_GRACE_SECONDS: son durum gorunsun diye disconnect oncesi kisa bekleme suresi.
    AUTO_DISCONNECT_GRACE_SECONDS = 20

    # AUTO_PUSH_TO_GITHUB: final exportlar bitince runs/<crop>/<part>/<RUN_ID>/ klasorunu repoya commit edip pushlar.
    AUTO_PUSH_TO_GITHUB = True

    # AUTO_PUSH_REMOTE_NAME: auto-push aciksa kullanilacak git remote adi.
    AUTO_PUSH_REMOTE_NAME = "origin"

    # AUTO_PUSH_BRANCH: auto-push icin branch override degeri. None olursa mevcut branch kullanilir.
    AUTO_PUSH_BRANCH = None

    # MANUAL_PARAM_OVERRIDES: anahtarlar notebook degisken adlariyla birebir ayni olmali.
    # Ornek: {"BATCH_SIZE": 32, "EPOCHS": 16, "PIN_MEMORY": False}
    MANUAL_PARAM_OVERRIDES = {}

    # Notebook icinde daha sonra kullanilacak gizli repo varsayimlarini yukle.
    BASE_CONFIG = ConfigurationManager(config_dir=str(ROOT / "config"), environment="colab").load_all_configs()
    CONTINUAL_DATA_CFG = BASE_CONFIG.get("training", {}).get("continual", {}).get("data", {})

    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = bool(TF32_ENABLED)

    TARGET_SIZE = int(CONTINUAL_DATA_CFG.get("target_size", 224))
    DATA_SAMPLER = str(CONTINUAL_DATA_CFG.get("sampler", "shuffle"))
    AUGMENTATION_POLICY = str(CONTINUAL_DATA_CFG.get("augmentation_policy", "randaugment")).strip().lower()
    RANDAUGMENT_NUM_OPS = int(CONTINUAL_DATA_CFG.get("randaugment_num_ops", 2))
    RANDAUGMENT_MAGNITUDE = int(CONTINUAL_DATA_CFG.get("randaugment_magnitude", 7))
    FEW_SHOT_RESEARCH_MODE = bool(FEW_SHOT_RESEARCH_MODE)
    FEW_SHOT_MIN_CLASS_SAMPLES = int(FEW_SHOT_MIN_CLASS_SAMPLES)
    LOADER_ERROR_POLICY = str(CONTINUAL_DATA_CFG.get("loader_error_policy", "tolerant"))
    CACHE_SIZE = int(CONTINUAL_DATA_CFG.get("cache_size", 1000))
    VALIDATE_IMAGES_ON_INIT = bool(CONTINUAL_DATA_CFG.get("validate_images_on_init", True))

    from scripts.colab_training_recommendations import resolve_effective_notebook_params

    def _collect_notebook_base_params():
        return {
            "FEW_SHOT_RESEARCH_MODE": bool(FEW_SHOT_RESEARCH_MODE),
            "FEW_SHOT_MIN_CLASS_SAMPLES": int(FEW_SHOT_MIN_CLASS_SAMPLES),
            "EPOCHS": int(EPOCHS),
            "BATCH_SIZE": int(BATCH_SIZE),
            "LEARNING_RATE": float(LEARNING_RATE),
            "LORA_R": int(LORA_R),
            "LORA_ALPHA": int(LORA_ALPHA),
            "LORA_DROPOUT": float(LORA_DROPOUT),
            "OOD_FACTOR": float(OOD_FACTOR),
            "SURE_SEMANTIC_PERCENTILE": float(SURE_SEMANTIC_PERCENTILE),
            "SURE_CONFIDENCE_PERCENTILE": float(SURE_CONFIDENCE_PERCENTILE),
            "CONFORMAL_ALPHA": float(CONFORMAL_ALPHA),
            "CONFORMAL_METHOD": str(CONFORMAL_METHOD),
            "CONFORMAL_RAPS_LAMBDA": float(CONFORMAL_RAPS_LAMBDA),
            "CONFORMAL_RAPS_K_REG": int(CONFORMAL_RAPS_K_REG),
            "BER_ENABLED": bool(BER_ENABLED),
            "BER_LAMBDA_OLD": float(BER_LAMBDA_OLD),
            "BER_LAMBDA_NEW": float(BER_LAMBDA_NEW),
            "BER_WARMUP_STEPS": int(BER_WARMUP_STEPS),
            "RANDAUGMENT_NUM_OPS": int(RANDAUGMENT_NUM_OPS),
            "RANDAUGMENT_MAGNITUDE": int(RANDAUGMENT_MAGNITUDE),
            "WEIGHT_DECAY": float(WEIGHT_DECAY),
            "MIXED_PRECISION": str(MIXED_PRECISION),
            "GRAD_ACCUM_STEPS": int(GRAD_ACCUM_STEPS),
            "MAX_GRAD_NORM": float(MAX_GRAD_NORM),
            "LABEL_SMOOTHING": float(LABEL_SMOOTHING),
            "LOSS_NAME": str(LOSS_NAME),
            "LOGITNORM_TAU": float(LOGITNORM_TAU),
            "SCHEDULER_NAME": str(SCHEDULER_NAME),
            "SCHEDULER_WARMUP_RATIO": float(SCHEDULER_WARMUP_RATIO),
            "SCHEDULER_MIN_LR": float(SCHEDULER_MIN_LR),
            "EARLY_STOPPING_PATIENCE": int(EARLY_STOPPING_PATIENCE),
            "EARLY_STOPPING_MIN_DELTA": float(EARLY_STOPPING_MIN_DELTA),
            "DETERMINISTIC": bool(DETERMINISTIC),
            "SEED": int(SEED),
            "NUM_WORKERS": int(NUM_WORKERS),
            "PREFETCH": int(PREFETCH),
            "PIN_MEMORY": bool(PIN_MEMORY),
            "USE_CACHE": bool(USE_CACHE),
            "CACHE_TRAIN_SPLIT": bool(CACHE_TRAIN_SPLIT),
            "VALIDATION_EVERY_N_EPOCHS": int(VALIDATION_EVERY_N_EPOCHS),
            "CHECKPOINT_EVERY_N_STEPS": int(CHECKPOINT_EVERY_N_STEPS),
            "CHECKPOINT_ON_EXCEPTION": bool(CHECKPOINT_ON_EXCEPTION),
            "STDOUT_BATCH_INTERVAL": int(STDOUT_BATCH_INTERVAL),
            "RESUME_MODE": str(RESUME_MODE),
            "AUTO_DISCONNECT_RUNTIME": bool(AUTO_DISCONNECT_RUNTIME),
            "AUTO_DISCONNECT_GRACE_SECONDS": int(AUTO_DISCONNECT_GRACE_SECONDS),
            "AUTO_PUSH_TO_GITHUB": bool(AUTO_PUSH_TO_GITHUB),
            "AUTO_PUSH_REMOTE_NAME": str(AUTO_PUSH_REMOTE_NAME),
            "AUTO_PUSH_BRANCH": AUTO_PUSH_BRANCH,
            "ENABLE_BAYESIAN_OPTIMIZATION": bool(ENABLE_BAYESIAN_OPTIMIZATION),
        }

    def _collect_bayesian_notebook_overrides():
        if not bool(ENABLE_BAYESIAN_OPTIMIZATION):
            return {}
        recommendations_path = ROOT / "runs" / "_index" / "bayesian_recommendations.json"
        if not recommendations_path.exists():
            print(f"[BAYES] Toggle acik ama dosya yok: {recommendations_path}")
            return {}
        try:
            payload = json.loads(recommendations_path.read_text(encoding="utf-8"))
        except Exception as exc:
            print(f"[BAYES] Oneri dosyasi okunamadi: {exc}")
            return {}

        cohorts = list(payload.get("cohorts", [])) if isinstance(payload, dict) else []
        selected_cohort = None
        for cohort in cohorts:
            comparability = cohort.get("comparability", {}) if isinstance(cohort, dict) else {}
            if str(comparability.get("crop_name", "")).strip().lower() == str(CROP_NAME).strip().lower() and str(comparability.get("part_name", "")).strip().lower() == str(PART_NAME).strip().lower():
                selected_cohort = cohort
                break
        if selected_cohort is None and cohorts:
            selected_cohort = cohorts[0]

        proposals = list((selected_cohort or {}).get("proposals", [])) if isinstance(selected_cohort, dict) else []
        if not proposals:
            print("[BAYES] Uygulanabilir oneriler bulunamadi.")
            return {}

        proposal = proposals[0]
        parameters = proposal.get("parameters", {}) if isinstance(proposal, dict) else {}
        if not isinstance(parameters, dict):
            return {}

        mapping = {
            "training.learning_rate": "LEARNING_RATE",
            "training.weight_decay": "WEIGHT_DECAY",
            "training.num_epochs": "EPOCHS",
            "training.batch_size": "BATCH_SIZE",
            "training.adapter.lora_r": "LORA_R",
            "training.adapter.lora_alpha": "LORA_ALPHA",
            "training.adapter.lora_dropout": "LORA_DROPOUT",
            "training.ood.threshold_factor": "OOD_FACTOR",
            "training.optimization.logitnorm_tau": "LOGITNORM_TAU",
            "training.data.randaugment_num_ops": "RANDAUGMENT_NUM_OPS",
            "training.data.randaugment_magnitude": "RANDAUGMENT_MAGNITUDE",
        }
        overrides = {}
        for source_key, target_key in mapping.items():
            if source_key in parameters:
                overrides[target_key] = parameters[source_key]

        rank = int(proposal.get("rank", 1)) if str(proposal.get("rank", "")).strip() else 1
        print(f"[BAYES] rank={rank} ile {len(overrides)} parametre onerisi yuklendi.")
        return overrides

    bayesian_overrides = _collect_bayesian_notebook_overrides()
    resolved_manual_overrides = dict(bayesian_overrides)
    resolved_manual_overrides.update(dict(MANUAL_PARAM_OVERRIDES or {}))

    INITIAL_EFFECTIVE_PARAMS = resolve_effective_notebook_params(
        _collect_notebook_base_params(),
        {"recommended_params": _collect_notebook_base_params()},
        resolved_manual_overrides,
        accepted=False,
    )

    STATE = {
        "validated": False,
        "class_names": [],
        "runtime_dataset_root": None,
        "runtime_dataset_key": None,
        "selected_dataset_name": None,
        "selected_dataset_root": None,
        "resolved_ood_root": None,
        "dataset_inspection": {},
        "hardware_inspection": {},
        "recommendation_report": {},
        "recommendation_decision": "pending",
        "effective_params": dict(INITIAL_EFFECTIVE_PARAMS),
        "adapter": None,
        "loaders": None,
        "history": None,
        "calibration": None,
        "checkpoint_manager": CHECKPOINT_MANAGER,
        "resume_manifest": None,
        "best_val_loss": None,
        "best_metric_state": {},
        "auto_disconnect_report": None,
        "git_push_report": None,
        "telemetry": TELEMETRY,
    }

    if str(INITIAL_EFFECTIVE_PARAMS.get("RESUME_MODE", RESUME_MODE)) == "resume":
        try:
            STATE["resume_manifest"] = CHECKPOINT_MANAGER.get_latest()
            if STATE["resume_manifest"]:
                manifest = STATE["resume_manifest"]
                print(
                    f"[RESUME] Checkpoint bulundu: {manifest.get('name', '?')} "
                    f"epoch={manifest.get('epoch', 0)} step={manifest.get('global_step', 0)}"
                )
        except Exception:
            pass

    hf_token_ready = False
    hf_token = str(resolve_hf_token() or "").strip()
    if not hf_token:
        raise RuntimeError(
            "Notebook 2 egitim baslamadan once Hugging Face token ister. "
            "HF_TOKEN, HUGGINGFACE_TOKEN veya HUGGINGFACE_HUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
        )
    hf_token_ready = bool(login_and_check_hf_token(print_fn=print))
    if not hf_token_ready:
        raise RuntimeError(
            "Hugging Face token on kontrolu basarisiz oldu. Egitimden once tokeni duzeltin."
        )

    github_token_ready = False
    if bool(INITIAL_EFFECTIVE_PARAMS.get("AUTO_PUSH_TO_GITHUB", AUTO_PUSH_TO_GITHUB)):
        github_token = str(resolve_github_token() or "").strip()
        if not github_token:
            raise RuntimeError(
                "AUTO_PUSH_TO_GITHUB is enabled, but no GitHub token was found. "
                "Egitim baslamadan once GH_TOKEN veya GITHUB_TOKEN degerini env var ya da Colab secret olarak tanimlayin."
            )
        github_token_ready = True
        print("[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.")

    print(
        f"[CONFIG] source=notebook_cell crop={CROP_NAME} epochs={EPOCHS} bs={BATCH_SIZE} "
        f"lr={LEARNING_RATE} resume={RESUME_MODE} ber={BER_ENABLED} "
        f"loss={LOSS_NAME} tau={LOGITNORM_TAU} auto_disconnect={AUTO_DISCONNECT_RUNTIME} "
        f"auto_push={AUTO_PUSH_TO_GITHUB} bayes_opt={ENABLE_BAYESIAN_OPTIMIZATION}"
    )
    print(
        f"[DATASET] runtime_root={RUNTIME_DATASET_ROOT} dataset_name={DATASET_NAME or '<ask>'}"
    )
    print(
        f"[OOD] ood_root={OOD_ROOT or '<ask>'} ask_for_ood_root={ASK_FOR_OOD_ROOT}"
    )
    print(
        f"[RUNTIME] defaults=notebook_cell mp={MIXED_PRECISION} workers={NUM_WORKERS} prefetch={PREFETCH} "
        f"sched={SCHEDULER_NAME} wd={WEIGHT_DECAY} accum={GRAD_ACCUM_STEPS} grad_clip={MAX_GRAD_NORM} "
        f"label_smooth={LABEL_SMOOTHING} warmup={SCHEDULER_WARMUP_RATIO} "
        f"early_stop={EARLY_STOPPING_PATIENCE}/{EARLY_STOPPING_MIN_DELTA} "
        f"val_every={VALIDATION_EVERY_N_EPOCHS} cache_train={CACHE_TRAIN_SPLIT} "
        f"aug={AUGMENTATION_POLICY} randaug={RANDAUGMENT_NUM_OPS}/{RANDAUGMENT_MAGNITUDE} "
        f"few_shot_research={FEW_SHOT_RESEARCH_MODE} min_class={FEW_SHOT_MIN_CLASS_SAMPLES}"
    )
    print(
        f"[OOD] factor={OOD_FACTOR} sure={SURE_SEMANTIC_PERCENTILE}/{SURE_CONFIDENCE_PERCENTILE} "
        f"conformal={CONFORMAL_METHOD} alpha={CONFORMAL_ALPHA} "
        f"raps_lambda={CONFORMAL_RAPS_LAMBDA} raps_k={CONFORMAL_RAPS_K_REG}"
    )
    TELEMETRY.update_latest(
        {
            "phase": "parameters_ready",
            "parameter_source": "notebook_cell",
            "crop": CROP_NAME,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "dataset_name": DATASET_NAME,
            "ood_root": OOD_ROOT,
            "ask_for_ood_root": ASK_FOR_OOD_ROOT,
            "loss_name": LOSS_NAME,
            "logitnorm_tau": LOGITNORM_TAU,
            "mixed_precision": MIXED_PRECISION,
            "num_workers": NUM_WORKERS,
            "prefetch": PREFETCH,
            "cache_train_split": CACHE_TRAIN_SPLIT,
            "augmentation_policy": AUGMENTATION_POLICY,
            "randaugment_num_ops": RANDAUGMENT_NUM_OPS,
            "randaugment_magnitude": RANDAUGMENT_MAGNITUDE,
            "few_shot_research_mode": FEW_SHOT_RESEARCH_MODE,
            "few_shot_min_class_samples": FEW_SHOT_MIN_CLASS_SAMPLES,
            "ber_enabled": BER_ENABLED,
            "ber_lambda_old": BER_LAMBDA_OLD,
            "ber_lambda_new": BER_LAMBDA_NEW,
            "ber_warmup_steps": BER_WARMUP_STEPS,
            "ood_factor": OOD_FACTOR,
            "sure_semantic_percentile": SURE_SEMANTIC_PERCENTILE,
            "sure_confidence_percentile": SURE_CONFIDENCE_PERCENTILE,
            "conformal_alpha": CONFORMAL_ALPHA,
            "conformal_method": CONFORMAL_METHOD,
            "conformal_raps_lambda": CONFORMAL_RAPS_LAMBDA,
            "conformal_raps_k_reg": CONFORMAL_RAPS_K_REG,
            "label_smoothing": LABEL_SMOOTHING,
            "scheduler_warmup_ratio": SCHEDULER_WARMUP_RATIO,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "early_stopping_min_delta": EARLY_STOPPING_MIN_DELTA,
            "resume_mode": RESUME_MODE,
            "validation_every_n_epochs": VALIDATION_EVERY_N_EPOCHS,
            "enable_bayesian_optimization": bool(ENABLE_BAYESIAN_OPTIMIZATION),
            "auto_push_to_github": AUTO_PUSH_TO_GITHUB,
            "hf_token_ready": hf_token_ready,
            "auto_push_token_ready": github_token_ready,
            "auto_push_remote_name": AUTO_PUSH_REMOTE_NAME,
            "auto_push_branch": AUTO_PUSH_BRANCH,
            "auto_disconnect_runtime": AUTO_DISCONNECT_RUNTIME,
            "auto_disconnect_grace_seconds": AUTO_DISCONNECT_GRACE_SECONDS,
        }
    )


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


[HF] Kimlik dogrulandi: Grizzmann
[GIT] Auto-push on kontrolu gecti: GitHub token cozuldu.
[CONFIG] source=notebook_cell crop=grape epochs=24 bs=32 lr=0.0001 resume=fresh ber=False loss=logitnorm tau=1.0 auto_disconnect=True auto_push=True bayes_opt=False
[DATASET] runtime_root=data/prepared_runtime_datasets/grape__leaf dataset_name=<ask>
[OOD] ood_root=data/ood_dataset/grape_leaf_ood_best ask_for_ood_root=True
[RUNTIME] defaults=notebook_cell mp=bf16 workers=12 prefetch=8 sched=cosine wd=0.02 accum=1 grad_clip=1.0 label_smooth=0.0 warmup=0.1 early_stop=5/0.0 val_every=1 cache_train=True aug=randaugment randaug=2/7 few_shot_research=False min_class=1
[OOD] factor=3.25 sure=90.0/97.0 conformal=raps alpha=0.05 raps_lambda=0.2 raps_k=1


In [11]:
with TELEMETRY.capture_cell_output("Cell 3b: Guncelleme ve Erisim Kontrolu"):
    from scripts.colab_repo_bootstrap import collect_notebook_access_report, print_notebook_access_report

    backbone_model = str(dict(BASE_CONFIG.get("training", {}).get("continual", {})).get("backbone", {}).get("model_name", "")).strip()
    access_report = collect_notebook_access_report(
        repo_root=ROOT,
        hf_model_ids=[backbone_model] if str(backbone_model).strip() else [],
    )
    STATE["access_report"] = access_report
    print_notebook_access_report(access_report, print_fn=print)
    TELEMETRY.update_latest(
        {
            "phase": "access_checked",
            "github_read_access": access_report.get("github", {}).get("read_access_mode"),
            "repo_update_relation": access_report.get("repo_updates", {}).get("relation"),
            "hf_access_mode": access_report.get("huggingface", {}).get("access_mode"),
        }
    )


[KONTROL] Repo guncel gorunuyor.
[KONTROL] GitHub okuma erisimi public; clone/pull icin ekstra token gerekmiyor.
[KONTROL] GitHub push icin kimlik bilgisi hazir.
[KONTROL] Gerekli Hugging Face modelleri anonim erisimle aciliyor.


In [12]:
with TELEMETRY.capture_cell_output("Cell 4: Dataset Validation"):
    from scripts.colab_dataset_layout import list_repo_dataset_directories, resolve_direct_repo_dataset_root, resolve_repo_relative_root
    from scripts.colab_training_recommendations import (
        inspect_runtime_dataset,
        inspect_runtime_hardware,
        recommend_notebook_training_params,
        resolve_effective_notebook_params,
    )

    crop_key = "".join(ch.lower() if ch.isalnum() else "_" for ch in str(CROP_NAME).strip())
    while "__" in crop_key:
        crop_key = crop_key.replace("__", "_")
    crop_key = crop_key.strip("_")
    if not crop_key:
        raise RuntimeError("CROP_NAME bos olmayan bir crop anahtarina cozulmeli.")

    runtime_parent = resolve_repo_relative_root(repo_root=ROOT, repo_relative_root=RUNTIME_DATASET_ROOT)
    direct_runtime_dataset = resolve_direct_repo_dataset_root(
        repo_root=ROOT,
        repo_relative_root=RUNTIME_DATASET_ROOT,
    )
    runtime_dirs = [] if direct_runtime_dataset is not None else list_repo_dataset_directories(
        repo_root=ROOT,
        repo_relative_root=RUNTIME_DATASET_ROOT,
    )
    candidates = []
    if direct_runtime_dataset is not None:
        direct_runtime_name, direct_runtime_path = direct_runtime_dataset
        candidates.append({"name": direct_runtime_name, "path": direct_runtime_path, "parent": runtime_parent})
    else:
        candidates.extend(
            {"name": path.name, "path": path, "parent": runtime_parent}
            for path in runtime_dirs
        )
    if not candidates:
        raise RuntimeError("No prepared runtime datasets were found under RUNTIME_DATASET_ROOT. Notebook 0'u once calistirin.")

    requested_dataset_name = str(DATASET_NAME).strip()
    if requested_dataset_name:
        matches = [item for item in candidates if item["name"] == requested_dataset_name]
        if not matches:
            available_options = [item['name'] for item in candidates]
            raise RuntimeError(
                f"Requested dataset '{requested_dataset_name}' was not found. Available options: {available_options}"
            )
        selected = matches[0]
    elif len(candidates) == 1:
        selected = candidates[0]
        print(f"[DATASET] Yalnizca bir runtime dataset bulundu, otomatik secildi: {selected['name']}")
    else:
        print("[DATASET] Kullanilabilir runtime dataset secenekleri:")
        for index, item in enumerate(candidates, start=1):
            print(f"  [{index}] {item['name']} ({item['path']})")
        raw_choice = str(input(f"Kullanilacak dataset icin numara ya da isim girin (1-{len(candidates)}): ")).strip()
        if not raw_choice:
            raise RuntimeError("Dataset secimi bos birakilamaz.")
        if raw_choice.isdigit():
            selected_index = int(raw_choice) - 1
            if selected_index < 0 or selected_index >= len(candidates):
                raise RuntimeError(f"Dataset secim index'i aralik disi: {raw_choice}")
            selected = candidates[selected_index]
        else:
            matches = [item for item in candidates if item['name'] == raw_choice]
            if not matches:
                raise RuntimeError(f"Dataset secimi bulunamadi: {raw_choice}")
            selected = matches[0]

    selected_dataset_name = str(selected['name'])
    selected_dataset_root = Path(selected['path'])
    if not selected_dataset_name.startswith(crop_key):
        raise RuntimeError(
            f"Secilen runtime dataset CROP_NAME ile uyusmuyor: {selected_dataset_name} vs {crop_key}"
        )
    missing_splits = [name for name in ("continual", "val", "test") if not (selected_dataset_root / name).is_dir()]
    if missing_splits:
        raise RuntimeError(f"Prepared runtime dataset is missing split folder(s): {missing_splits}")
    class_names = sorted(d.name for d in (selected_dataset_root / "continual").iterdir() if d.is_dir())
    if not class_names:
        raise RuntimeError(f"No class subdirectories in prepared runtime split: {selected_dataset_root / 'continual'}")
    runtime_root = selected_dataset_root.parent
    default_ood_root = selected_dataset_root / "ood"
    requested_ood_root = str(OOD_ROOT or "").strip()
    if bool(ASK_FOR_OOD_ROOT) and not requested_ood_root:
        default_hint = str(default_ood_root) if default_ood_root.is_dir() else ""
        prompt = "OOD klasoru yolunu girin"
        if default_hint:
            prompt += f" [Enter={default_hint}]"
        requested_ood_root = str(input(prompt + ": ")).strip()
        if not requested_ood_root and default_hint:
            requested_ood_root = default_hint

    if requested_ood_root:
        resolved_ood_root = Path(requested_ood_root).expanduser()
        if not resolved_ood_root.is_absolute():
            resolved_ood_root = (ROOT / resolved_ood_root).resolve()
        if not resolved_ood_root.is_dir():
            raise RuntimeError(f"OOD klasoru bulunamadi veya klasor degil: {resolved_ood_root}")
        print(f"[OOD] explicit ood root={resolved_ood_root}")
        resolved_ood_root_value = str(resolved_ood_root)
    elif default_ood_root.is_dir():
        print(f"[OOD] runtime ood root={default_ood_root}")
        resolved_ood_root_value = str(default_ood_root)
    else:
        print("[OOD] Gercek OOD split secilmedi; fallback held-out benchmark kullanilabilir.")
        resolved_ood_root_value = ""
    print(f"[DATASET] runtime root={selected_dataset_root} classes={len(class_names)}: {class_names}")

    base_params = _collect_notebook_base_params()
    dataset_inspection = inspect_runtime_dataset(selected_dataset_root, ood_root=resolved_ood_root_value or None)
    hardware_inspection = inspect_runtime_hardware(DEVICE)
    recommendation_report = recommend_notebook_training_params(base_params, dataset_inspection, hardware_inspection)

    split_totals = dict(dataset_inspection.get("split_totals", {}))
    print(
        f"[RECOMMEND][DATASET] scale={dataset_inspection.get('dataset_scale_bucket', 'unknown')} "
        f"continual={split_totals.get('continual', 0)} val={split_totals.get('val', 0)} "
        f"test={split_totals.get('test', 0)} ood={split_totals.get('ood', 0)} "
        f"classes={dataset_inspection.get('class_count', 0)}"
    )
    print(
        f"[RECOMMEND][HW] device={hardware_inspection.get('effective_device', 'cpu')} "
        f"gpu={hardware_inspection.get('gpu_name') or 'none'} "
        f"vram_gb={hardware_inspection.get('total_vram_gb')} cpu_count={hardware_inspection.get('cpu_count', 0)}"
    )

    blockers = list(recommendation_report.get("blockers", []))
    warnings = list(recommendation_report.get("warnings", []))
    for item in warnings:
        print(f"[RECOMMEND][WARN] {item}")
    for item in blockers:
        print(f"[RECOMMEND][BLOCK] {item}")

    accepted_recommendations = False
    recommendation_decision = "no_changes"
    if recommendation_report.get("has_changes"):
        print("[RECOMMEND] Onerilen degisiklikler:")
        for key in sorted(recommendation_report.get("changes", {})):
            change = recommendation_report["changes"][key]
            print(f"  - {key}: {change['current']} -> {change['recommended']} | {change['reason']}")
        if blockers:
            recommendation_decision = "blocked"
            print("[RECOMMEND] Blocker oldugu icin otomatik uygulama sorusu atlandi. Gerekirse MANUAL_PARAM_OVERRIDES ile acik override girin.")
        else:
            raw_confirm = str(input("Apply recommended parameters? [y/N]: ")).strip().lower()
            accepted_recommendations = raw_confirm in {"y", "yes"}
            recommendation_decision = "accepted" if accepted_recommendations else "rejected"
    elif blockers:
        recommendation_decision = "blocked"
        print("[RECOMMEND] Blocker var; notebook ham parametrelerle devam eder. Gerekirse MANUAL_PARAM_OVERRIDES guncellenmeli.")
    else:
        print("[RECOMMEND] Mevcut notebook parametreleri onerilen degerlerle zaten uyumlu.")

    effective_params = resolve_effective_notebook_params(
        base_params,
        recommendation_report,
        MANUAL_PARAM_OVERRIDES,
        accepted=accepted_recommendations,
    )
    if MANUAL_PARAM_OVERRIDES:
        print(f"[RECOMMEND] Manual overrides uygulandi: {MANUAL_PARAM_OVERRIDES}")
    print(
        f"[RECOMMEND][FINAL] epochs={effective_params['EPOCHS']} bs={effective_params['BATCH_SIZE']} "
        f"lr={effective_params['LEARNING_RATE']} lora={effective_params['LORA_R']}/{effective_params['LORA_ALPHA']} "
        f"dropout={effective_params['LORA_DROPOUT']} accum={effective_params['GRAD_ACCUM_STEPS']} "
        f"workers={effective_params['NUM_WORKERS']} prefetch={effective_params['PREFETCH']}"
    )

    STATE["class_names"] = class_names
    STATE["validated"] = True
    STATE["runtime_dataset_root"] = runtime_root
    STATE["runtime_dataset_key"] = selected_dataset_name
    STATE["selected_dataset_name"] = selected_dataset_name
    STATE["selected_dataset_root"] = selected_dataset_root
    STATE["resolved_ood_root"] = resolved_ood_root_value
    STATE["dataset_inspection"] = dataset_inspection
    STATE["hardware_inspection"] = hardware_inspection
    STATE["recommendation_report"] = recommendation_report
    STATE["recommendation_decision"] = recommendation_decision
    STATE["effective_params"] = effective_params
    TELEMETRY.update_latest(
        {
            "phase": "dataset_validated",
            "dataset_root": str(selected_dataset_root),
            "runtime_dataset_root": str(runtime_root),
            "runtime_dataset_key": selected_dataset_name,
            "selected_dataset_name": selected_dataset_name,
            "resolved_ood_root": resolved_ood_root_value,
            "class_count": len(class_names),
            "recommendation_decision": recommendation_decision,
            "recommendation_change_count": int(recommendation_report.get('change_count', 0)),
        }
    )


[DATASET] Yalnizca bir runtime dataset bulundu, otomatik secildi: grape__leaf
[OOD] explicit ood root=/content/bitirmeprojesi/data/ood_dataset/grape_leaf_ood_best
[DATASET] runtime root=/content/bitirmeprojesi/data/prepared_runtime_datasets/grape__leaf classes=6: ['downy_mildew', 'esca', 'grapevine_fanleaf_virus', 'healthy', 'leafroll_virus', 'powdery_mildew']
[RECOMMEND][DATASET] scale=small continual=839 val=216 test=216 ood=209 classes=6
[RECOMMEND][HW] device=cuda gpu=NVIDIA A100-SXM4-80GB vram_gb=79.25 cpu_count=12
[RECOMMEND][WARN] Manifest rows include 174 eval-quality-risk item(s); prefer cautious regularization.
[RECOMMEND] Onerilen degisiklikler:
  - BATCH_SIZE: 32 -> 96 | Derived from the active hardware tier (device=cuda, vram_gb=79.25).
  - EPOCHS: 24 -> 16 | Uses the fixed schedule for a small continual split (839 image(s)).
  - NUM_WORKERS: 12 -> 6 | Derived from device=cuda and cpu_count=12.
Apply recommended parameters? [y/N]: N
[RECOMMEND][FINAL] epochs=24 bs=32 lr=0.

In [13]:
with TELEMETRY.capture_cell_output("Cell 5: Engine Init"):
    from scripts.colab_dataset_layout import resolve_notebook_training_classes
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.data.loaders import create_training_loaders

    def _normalize(name: str) -> str:
        normalized = "".join(ch.lower() if ch.isalnum() else "_" for ch in name.strip())
        while "__" in normalized:
            normalized = normalized.replace("__", "_")
        return normalized.strip("_")

    if not STATE.get("validated"):
        raise RuntimeError("Once dataset validation hucresini calistirin.")

    effective_params = dict(STATE.get("effective_params") or {})
    if not effective_params:
        raise RuntimeError("Effective notebook parameters cozulmedi. Once dataset validation hucresini yeniden calistirin.")

    crop_name = CROP_NAME.strip().lower()
    dataset_key = str(STATE.get("runtime_dataset_key") or DATASET_NAME).strip()
    if not dataset_key:
        raise RuntimeError("Runtime dataset key cozulmedi. Once dataset validation hucresini calistirin.")
    runtime_data_root = Path(STATE.get("runtime_dataset_root") or RUNTIME_DATASET_ROOT).expanduser()
    if not runtime_data_root.is_absolute():
        runtime_data_root = (ROOT / runtime_data_root).resolve()
    class_root = runtime_data_root / dataset_key / "continual"
    if not class_root.is_dir():
        raise RuntimeError(f"Prepared runtime continual split not found: {class_root}")
    available = sorted({_normalize(path.name) for path in class_root.iterdir() if path.is_dir() and _normalize(path.name)})
    resolved_ood_root = str(STATE.get("resolved_ood_root") or "").strip()
    if resolved_ood_root:
        print(f"[OOD] selected ood root={resolved_ood_root}")

    class_resolution = resolve_notebook_training_classes(
        available_classes=available,
        crop_name=crop_name,
        taxonomy_path=ROOT / "config" / "plant_taxonomy.json",
    )
    aligned = list(class_resolution.get("selected_classes", available))

    if not aligned:
        raise RuntimeError(f"No usable classes for crop '{crop_name}'. Available: {available}")

    final_class_names = aligned
    STATE["runtime_dataset_root"] = runtime_data_root
    STATE["runtime_dataset_key"] = dataset_key

    STATE["class_names"] = final_class_names
    STATE["class_resolution"] = class_resolution
    print(f"[CLASSES] {final_class_names}")
    print(
        f"[CLASSES] mode={'taxonomy_filter' if class_resolution.get('used_taxonomy_filter') else 'dataset_fallback'} "
        f"reason={class_resolution.get('reason', 'unknown')} "
        f"matched={len(class_resolution.get('matched_classes', []))} "
        f"unmatched={len(class_resolution.get('unmatched_classes', []))}"
    )
    if class_resolution.get("unmatched_classes"):
        print(f"[CLASSES] taxonomy-unmatched classes kept: {class_resolution['unmatched_classes']}")

    continual_cfg = json.loads(json.dumps(BASE_CONFIG.get("training", {}).get("continual", {})))
    continual_cfg["device"] = DEVICE
    continual_cfg["num_epochs"] = int(effective_params["EPOCHS"])
    continual_cfg["batch_size"] = int(effective_params["BATCH_SIZE"])
    continual_cfg["learning_rate"] = float(effective_params["LEARNING_RATE"])
    continual_cfg["adapter"]["lora_r"] = int(effective_params["LORA_R"])
    continual_cfg["adapter"]["lora_alpha"] = int(effective_params["LORA_ALPHA"])
    continual_cfg["adapter"]["lora_dropout"] = float(effective_params["LORA_DROPOUT"])
    continual_cfg["weight_decay"] = float(effective_params["WEIGHT_DECAY"])
    continual_cfg["deterministic"] = bool(effective_params["DETERMINISTIC"])
    data_cfg = continual_cfg.setdefault("data", {})
    data_cfg["augmentation_policy"] = str(AUGMENTATION_POLICY)
    data_cfg["randaugment_num_ops"] = int(effective_params["RANDAUGMENT_NUM_OPS"])
    data_cfg["randaugment_magnitude"] = int(effective_params["RANDAUGMENT_MAGNITUDE"])
    data_cfg["few_shot_research_mode"] = bool(effective_params["FEW_SHOT_RESEARCH_MODE"])
    data_cfg["few_shot_min_class_samples"] = int(effective_params["FEW_SHOT_MIN_CLASS_SAMPLES"])
    continual_cfg["ood"]["threshold_factor"] = float(effective_params["OOD_FACTOR"])
    continual_cfg["ood"]["sure_semantic_percentile"] = float(effective_params["SURE_SEMANTIC_PERCENTILE"])
    continual_cfg["ood"]["sure_confidence_percentile"] = float(effective_params["SURE_CONFIDENCE_PERCENTILE"])
    continual_cfg["ood"]["conformal_alpha"] = float(effective_params["CONFORMAL_ALPHA"])
    continual_cfg["ood"]["conformal_method"] = str(effective_params["CONFORMAL_METHOD"])
    continual_cfg["ood"]["conformal_raps_lambda"] = float(effective_params["CONFORMAL_RAPS_LAMBDA"])
    continual_cfg["ood"]["conformal_raps_k_reg"] = int(effective_params["CONFORMAL_RAPS_K_REG"])
    continual_cfg["ood"]["ber_enabled"] = bool(effective_params["BER_ENABLED"])
    continual_cfg["ood"]["ber_lambda_old"] = float(effective_params["BER_LAMBDA_OLD"])
    continual_cfg["ood"]["ber_lambda_new"] = float(effective_params["BER_LAMBDA_NEW"])
    continual_cfg["ood"]["ber_warmup_steps"] = int(effective_params["BER_WARMUP_STEPS"])
    print(f"[ENGINE][OOD_CFG] {json.dumps(continual_cfg['ood'], sort_keys=True)}")

    optimization_cfg = continual_cfg.setdefault("optimization", {})
    optimization_cfg["grad_accumulation_steps"] = int(effective_params["GRAD_ACCUM_STEPS"])
    optimization_cfg["max_grad_norm"] = float(effective_params["MAX_GRAD_NORM"])
    optimization_cfg["mixed_precision"] = str(effective_params["MIXED_PRECISION"])
    optimization_cfg["label_smoothing"] = float(effective_params["LABEL_SMOOTHING"])
    optimization_cfg["loss_name"] = str(effective_params["LOSS_NAME"]).strip().lower()
    optimization_cfg["logitnorm_tau"] = float(effective_params["LOGITNORM_TAU"])
    scheduler_cfg = optimization_cfg.setdefault("scheduler", {})
    scheduler_cfg["name"] = str(effective_params["SCHEDULER_NAME"])
    scheduler_cfg["warmup_ratio"] = float(effective_params["SCHEDULER_WARMUP_RATIO"])
    scheduler_cfg["min_lr"] = float(effective_params["SCHEDULER_MIN_LR"])
    scheduler_cfg["step_on"] = str(scheduler_cfg.get("step_on", "batch"))
    early_stopping_cfg = continual_cfg.setdefault("early_stopping", {})
    early_stopping_cfg["enabled"] = True
    early_stopping_cfg["patience"] = int(effective_params["EARLY_STOPPING_PATIENCE"])
    early_stopping_cfg["min_delta"] = float(effective_params["EARLY_STOPPING_MIN_DELTA"])
    print(
        f"[ENGINE][OPT_CFG] loss={optimization_cfg['loss_name']} tau={optimization_cfg['logitnorm_tau']} "
        f"label_smoothing={optimization_cfg['label_smoothing']} mixed_precision={optimization_cfg['mixed_precision']}"
    )

    adapter = IndependentCropAdapter(crop_name=crop_name, device=DEVICE)
    if hasattr(adapter, "part_name"):
        adapter.part_name = str(PART_NAME or "unspecified").strip().lower() or "unspecified"
    print("[ENGINE] Initializing adapter (may download backbone)...")
    adapter.initialize_engine(class_names=STATE["class_names"], config={"training": {"continual": continual_cfg}})

    loader_kwargs = {}
    if int(effective_params["NUM_WORKERS"]) > 0:
        loader_kwargs["prefetch_factor"] = int(effective_params["PREFETCH"])

    loaders = create_training_loaders(
        data_dir=str(runtime_data_root),
        crop=dataset_key,
        batch_size=int(effective_params["BATCH_SIZE"]),
        num_workers=int(effective_params["NUM_WORKERS"]),
        use_cache=bool(effective_params["USE_CACHE"]),
        cache_size=CACHE_SIZE,
        cache_train_split=bool(effective_params["CACHE_TRAIN_SPLIT"]),
        target_size=TARGET_SIZE,
        error_policy=LOADER_ERROR_POLICY,
        sampler=DATA_SAMPLER,
        seed=int(effective_params["SEED"]),
        validate_images_on_init=VALIDATE_IMAGES_ON_INIT,
        augmentation_policy=AUGMENTATION_POLICY,
        randaugment_num_ops=int(effective_params["RANDAUGMENT_NUM_OPS"]),
        randaugment_magnitude=int(effective_params["RANDAUGMENT_MAGNITUDE"]),
        ood_root=resolved_ood_root or None,
        pin_memory=bool(effective_params["PIN_MEMORY"]),
        **loader_kwargs,
    )

    STATE["crop_name"] = crop_name
    STATE["part_name"] = str(PART_NAME or "unspecified").strip().lower() or "unspecified"
    STATE["adapter"] = adapter
    STATE["loaders"] = loaders
    STATE["continual_config"] = continual_cfg

    trainable = sum(parameter.numel() for parameter in adapter.parameters() if parameter.requires_grad)
    print(f"[ENGINE] Hazir. trainable_params={trainable:,}  classes={len(final_class_names)}")
    TELEMETRY.update_latest(
        {
            "phase": "engine_ready",
            "class_count": len(final_class_names),
            "runtime_dataset_root": str(runtime_data_root),
            "runtime_dataset_key": dataset_key,
            "selected_dataset_name": str(STATE.get("selected_dataset_name") or ""),
            "resolved_ood_root": str(STATE.get("resolved_ood_root") or ""),
            "effective_batch_size": int(effective_params["BATCH_SIZE"]) * int(effective_params["GRAD_ACCUM_STEPS"]),
        }
    )


[OOD] selected ood root=/content/bitirmeprojesi/data/ood_dataset/grape_leaf_ood_best
[CLASSES] ['downy_mildew', 'esca', 'grapevine_fanleaf_virus', 'healthy', 'leafroll_virus', 'powdery_mildew']
[CLASSES] mode=dataset_fallback reason=partial_taxonomy_alignment_fallback matched=5 unmatched=1
[CLASSES] taxonomy-unmatched classes kept: ['leafroll_virus']
[ENGINE][OOD_CFG] {"ber_enabled": false, "ber_lambda_new": 0.1, "ber_lambda_old": 0.1, "ber_warmup_steps": 50, "conformal_alpha": 0.05, "conformal_enabled": true, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "energy_temperature": 1.0, "energy_temperature_mode": "fixed", "energy_temperature_range": [0.5, 3.0], "energy_temperature_steps": 16, "knn_backend": "auto", "knn_chunk_size": 2048, "primary_score_method": "auto", "radial_beta_range": [0.5, 2.0], "radial_beta_steps": 16, "radial_l2_enabled": true, "sure_confidence_percentile": 97.0, "sure_enabled": true, "sure_semantic_percentile": 90.0, "thresho

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

[ENGINE] Hazir. trainable_params=10,232,842  classes=6


In [14]:
with TELEMETRY.capture_cell_output("Cell 5b: OOD Config Verify"):
    if "continual_config" not in STATE:
        raise RuntimeError("Once Cell 5 calistirilmali ki continual_config hazir olsun.")

    effective_params = dict(STATE.get("effective_params") or {})
    resolved_ood_cfg = dict(STATE["continual_config"].get("ood", {}))
    expected_ood_cfg = {
        "threshold_factor": float(effective_params["OOD_FACTOR"]),
        "sure_semantic_percentile": float(effective_params["SURE_SEMANTIC_PERCENTILE"]),
        "sure_confidence_percentile": float(effective_params["SURE_CONFIDENCE_PERCENTILE"]),
        "conformal_alpha": float(effective_params["CONFORMAL_ALPHA"]),
        "conformal_method": str(effective_params["CONFORMAL_METHOD"]),
        "conformal_raps_lambda": float(effective_params["CONFORMAL_RAPS_LAMBDA"]),
        "conformal_raps_k_reg": int(effective_params["CONFORMAL_RAPS_K_REG"]),
    }

    print("[VERIFY][OOD][expected]", json.dumps(expected_ood_cfg, sort_keys=True))
    print("[VERIFY][OOD][resolved]", json.dumps({k: resolved_ood_cfg.get(k) for k in expected_ood_cfg}, sort_keys=True))

    mismatches = []
    for key, expected in expected_ood_cfg.items():
        actual = resolved_ood_cfg.get(key)
        if isinstance(expected, float):
            try:
                actual_float = float(actual)
            except Exception:
                mismatches.append(f"{key}: expected={expected} actual={actual}")
                continue
            if abs(actual_float - expected) > 1e-12:
                mismatches.append(f"{key}: expected={expected} actual={actual_float}")
        elif actual != expected:
            mismatches.append(f"{key}: expected={expected} actual={actual}")

    if mismatches:
        raise RuntimeError("OOD config mismatch:\n - " + "\n - ".join(mismatches))

    print("[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.")

[VERIFY][OOD][expected] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.25}
[VERIFY][OOD][resolved] {"conformal_alpha": 0.05, "conformal_method": "raps", "conformal_raps_k_reg": 1, "conformal_raps_lambda": 0.2, "sure_confidence_percentile": 97.0, "sure_semantic_percentile": 90.0, "threshold_factor": 3.25}
[VERIFY][OOD] OK: cozulmus OOD ayari istenen parametrelerle eslesiyor.


In [15]:
with TELEMETRY.capture_cell_output("Cell 6: Training"):
    from scripts.colab_notebook_helpers import (
        build_history_snapshot,
        persist_training_curve_figure,
        persist_training_history_artifacts,
        save_notebook_checkpoint,
    )

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Once engine init hucresini calistirin.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    checkpoint_manager = STATE["checkpoint_manager"]
    effective_params = dict(STATE.get("effective_params") or {})
    val_loader = loaders.get("val")
    run_id = RUN_ID
    telemetry = STATE.get("telemetry") or TELEMETRY

    resume_state = None
    resume_mode = str(effective_params.get("RESUME_MODE", "fresh"))
    resume_manifest = STATE.get("resume_manifest")
    if resume_mode == "resume" and not isinstance(resume_manifest, dict):
        try:
            resume_manifest = checkpoint_manager.get_latest()
            STATE["resume_manifest"] = resume_manifest
        except Exception:
            resume_manifest = None
    if resume_mode == "resume" and isinstance(resume_manifest, dict):
        checkpoint_path = str(resume_manifest.get("path", "")).strip()
        if checkpoint_path:
            try:
                resume_state = adapter.load_training_checkpoint(checkpoint_path)
                STATE["resume_state"] = resume_state
                progress_state = resume_state.get("progress_state") or {}
                print(
                    f"[RESUME] epoch={progress_state.get('epoch', 0)} "
                    f"step={progress_state.get('global_step', 0)}"
                )
            except Exception as exc:
                print(f"[RESUME] Basarisiz: {exc}")

    existing_history = (resume_state or {}).get("history", (resume_state or {}).get("history_snapshot", {}))
    train_loss_curve = list(existing_history.get("train_loss", []))
    val_loss_curve = list(existing_history.get("val_loss", []))
    val_acc_curve = list(existing_history.get("val_accuracy", []))
    macro_f1_curve = list(existing_history.get("macro_f1", []))
    weighted_f1_curve = list(existing_history.get("weighted_f1", []))
    balanced_acc_curve = list(existing_history.get("balanced_accuracy", []))
    gap_curve = list(existing_history.get("generalization_gap", []))

    start_time = time.time()
    session = None
    last_checkpoint_step = -1
    best_val_loss = float(STATE["best_val_loss"]) if STATE.get("best_val_loss") is not None else None

    print(f"[TRAIN] epochs={effective_params['EPOCHS']} device={DEVICE} batch_interval={effective_params['STDOUT_BATCH_INTERVAL']}")
    telemetry.update_latest(
        {
            "phase": "training_started",
            "epochs": int(effective_params["EPOCHS"]),
            "batch_interval": int(effective_params["STDOUT_BATCH_INTERVAL"]),
        }
    )

    def _history_snapshot():
        return build_history_snapshot(
            state_history=STATE.get("history"),
            train_loss_curve=train_loss_curve,
            val_loss_curve=val_loss_curve,
            val_acc_curve=val_acc_curve,
            macro_f1_curve=macro_f1_curve,
            weighted_f1_curve=weighted_f1_curve,
            balanced_acc_curve=balanced_acc_curve,
            gap_curve=gap_curve,
        )

    def _persist_history():
        snapshot = _history_snapshot()
        STATE["history"] = dict((STATE.get("history") or {}), **snapshot)
        persist_training_history_artifacts(
            root=ROOT,
            history_snapshot=STATE["history"],
            telemetry=telemetry,
        )
        return snapshot

    def _checkpoint(reason, event, mark_best=False, val_loss=None):
        if session is None:
            return None
        record = save_notebook_checkpoint(
            checkpoint_manager=checkpoint_manager,
            adapter=adapter,
            session=session,
            reason=reason,
            run_id=run_id,
            telemetry=telemetry,
            mark_best=bool(mark_best),
            val_loss=(float(val_loss) if val_loss is not None else None),
        )
        if record is not None:
            STATE["resume_manifest"] = record
            print(f"[CKPT] {reason} epoch={record.get('epoch', '?')} step={record.get('global_step', '?')}")
        return record

    def session_observer(record):
        global last_checkpoint_step, best_val_loss
        event_type = record.get("event_type", "")
        event = record.get("payload", {})

        if event_type == "stop_requested":
            return

        if event_type == "batch_end":
            batch_num = int(event.get("batch", 0))
            if batch_num > 0 and (batch_num % int(effective_params["STDOUT_BATCH_INTERVAL"]) == 0):
                print(
                    f"[TRAIN] epoch={event.get('epoch', 0)} "
                    f"batch={batch_num}/{event.get('total_batches', 0)} "
                    f"loss={event.get('batch_loss', 0.0):.4f} "
                    f"lr={event.get('lr', 0.0):.6f} "
                    f"speed={event.get('samples_per_sec', 0.0):.1f}s/s "
                    f"elapsed={event.get('elapsed_sec', 0.0):.0f}s eta={event.get('eta_sec', 0.0):.0f}s"
                )
            step = int(event.get("global_step", 0))
            if (
                int(effective_params["CHECKPOINT_EVERY_N_STEPS"]) > 0
                and step > 0
                and (step % int(effective_params["CHECKPOINT_EVERY_N_STEPS"]) == 0)
                and step != last_checkpoint_step
            ):
                _checkpoint(f"batch_{int(effective_params['CHECKPOINT_EVERY_N_STEPS'])}", event)
                last_checkpoint_step = step

        if event_type == "epoch_end":
            train_loss_curve.append(float(event.get("epoch_loss", 0.0)))
            for key, curve in [
                ("val_loss", val_loss_curve),
                ("val_accuracy", val_acc_curve),
                ("macro_f1", macro_f1_curve),
                ("weighted_f1", weighted_f1_curve),
                ("balanced_accuracy", balanced_acc_curve),
                ("generalization_gap", gap_curve),
            ]:
                if key in event:
                    curve.append(float(event[key]))

            val_loss = float(event["val_loss"]) if "val_loss" in event else None
            mark_best = False
            if val_loss is not None and (best_val_loss is None or val_loss < best_val_loss):
                best_val_loss = val_loss
                STATE["best_val_loss"] = best_val_loss
                mark_best = True

            should_persist_curve = (
                mark_best
                or int(event["epoch_done"]) == 1
                or int(event["epoch_done"]) == int(effective_params["EPOCHS"])
                or bool(event.get("stopped_early", False))
                or (int(event["epoch_done"]) % 5 == 0)
            )
            if should_persist_curve:
                plt.figure(figsize=(13, 3))
                plt.subplot(1, 3, 1)
                plt.plot(range(1, len(train_loss_curve) + 1), train_loss_curve, marker="o", label="Train")
                if val_loss_curve:
                    plt.plot(range(1, len(val_loss_curve) + 1), val_loss_curve, marker="s", label="Val")
                plt.xlabel("Epoch")
                plt.ylabel("Loss")
                plt.title("Loss")
                plt.grid(True, alpha=0.3)
                plt.legend()

                plt.subplot(1, 3, 2)
                for values, label, marker in [
                    (val_acc_curve, "Acc", "^"),
                    (macro_f1_curve, "MacroF1", "d"),
                    (weighted_f1_curve, "WtdF1", "x"),
                    (balanced_acc_curve, "BalAcc", "*"),
                ]:
                    if values:
                        plt.plot(range(1, len(values) + 1), values, marker=marker, label=label)
                plt.ylim(0, 1)
                plt.xlabel("Epoch")
                plt.ylabel("Score")
                plt.title("Metrics")
                plt.grid(True, alpha=0.3)
                plt.legend()

                plt.subplot(1, 3, 3)
                if gap_curve:
                    plt.plot(range(1, len(gap_curve) + 1), gap_curve, marker="o", label="Gap")
                plt.axhline(0, color="black", lw=1, alpha=0.5)
                plt.xlabel("Epoch")
                plt.ylabel("Gap")
                plt.title("Gen. Gap")
                plt.grid(True, alpha=0.3)
                plt.legend()
                plt.tight_layout()
                persist_training_curve_figure(
                    root=ROOT,
                    epoch_done=int(event["epoch_done"]),
                    telemetry=telemetry,
                )
                plt.close("all")

            _persist_history()

            if mark_best or bool(event.get("stopped_early", False)) or int(event["epoch_done"]) == int(effective_params["EPOCHS"]) or int(effective_params["CHECKPOINT_EVERY_N_STEPS"]) <= 0:
                _checkpoint("epoch_end", event, mark_best=mark_best, val_loss=val_loss)

            parts = [f"[EPOCH] {event['epoch_done']}/{effective_params['EPOCHS']}: train_loss={event.get('epoch_loss', 0.0):.4f}"]
            if "val_loss" in event:
                parts.append(f"val_loss={event['val_loss']:.4f}")
            if "val_accuracy" in event:
                parts.append(f"val_acc={event['val_accuracy']:.4f}")
            if "macro_f1" in event:
                parts.append(f"macro_f1={event['macro_f1']:.4f}")
            if mark_best:
                parts.append("* BEST")
            print(" ".join(parts))
            telemetry.update_latest(
                {
                    "phase": "training",
                    "epoch_done": int(event["epoch_done"]),
                    "global_step": int(event.get("global_step", 0)),
                    "best_val_loss": best_val_loss,
                }
            )

    session = adapter.build_training_session(
        loaders["train"],
        num_epochs=int(effective_params["EPOCHS"]),
        val_loader=val_loader,
        observers=[session_observer],
        resume_state=resume_state,
        run_id=run_id,
        validation_every_n_epochs=int(effective_params["VALIDATION_EVERY_N_EPOCHS"]),
    )

    try:
        history = session.run()
        adapter.is_trained = True
    except KeyboardInterrupt:
        print("[TRAIN] Stop requested by user. Saving latest checkpoint and progress snapshot...")
        telemetry.emit_log("Training stopped by user. Persisting checkpoint.", phase="train", level="warning")
        try:
            STATE["resume_state"] = session.snapshot_state()
        except Exception:
            pass
        if bool(effective_params["CHECKPOINT_ON_EXCEPTION"]):
            try:
                _checkpoint(
                    "manual_stop",
                    {
                        "epoch": 0,
                        "batch": 0,
                        "global_step": int((STATE.get("history") or {}).get("global_step", 0)),
                        "elapsed_sec": time.time() - start_time,
                    },
                )
            except Exception:
                pass
        raise
    except Exception as exc:
        print(f"[TRAIN] Exception: {exc}")
        telemetry.emit_log(f"Training exception: {exc}", phase="train", level="error")
        if bool(effective_params["CHECKPOINT_ON_EXCEPTION"]):
            try:
                _checkpoint(
                    "exception",
                    {
                        "epoch": 0,
                        "batch": 0,
                        "global_step": int((STATE.get("history") or {}).get("global_step", 0)),
                        "elapsed_sec": time.time() - start_time,
                    },
                )
            except Exception:
                pass
        raise

    elapsed_total = time.time() - start_time
    STATE["resume_state"] = session.snapshot_state()
    if history is not None:
        STATE["history"] = history.to_dict()
    elif STATE.get("history") is None:
        STATE["history"] = {}
    _persist_history()
    telemetry.update_latest(
        {
            "phase": "training_complete",
            "elapsed_sec": round(elapsed_total, 3),
            "stopped_early": bool(STATE["history"].get("stopped_early", False)),
            "stopped_by_user": False,
            "continuous_mode": False,
            "completed_cycles": 1,
        }
    )
    print(
        f"[TRAIN] Complete. elapsed={elapsed_total:.1f}s "
        f"stopped_early={STATE['history'].get('stopped_early', False)} "
        f"stopped_by_user=False cycles=1"
    )

[TRAIN] epochs=24 device=cuda batch_interval=12
[TRAIN] epoch=1 batch=12/26 loss=0.0000 lr=0.000100 speed=215.0s/s elapsed=5s eta=277s
[TRAIN] epoch=1 batch=24/26 loss=0.0000 lr=0.000100 speed=231.6s/s elapsed=7s eta=180s
[CKPT] epoch_end epoch=1 step=26
[EPOCH] 1/24: train_loss=1.8217 val_loss=1.8310 val_acc=0.1204 macro_f1=0.0358 * BEST
[TRAIN] epoch=2 batch=12/26 loss=0.0000 lr=0.000099 speed=232.2s/s elapsed=15s eta=224s
[TRAIN] epoch=2 batch=24/26 loss=0.0000 lr=0.000098 speed=228.2s/s elapsed=16s eta=187s
[CKPT] epoch_end epoch=2 step=52
[EPOCH] 2/24: train_loss=1.7889 val_loss=1.7336 val_acc=0.2917 macro_f1=0.1377 * BEST
[TRAIN] epoch=3 batch=12/26 loss=0.0000 lr=0.000097 speed=229.7s/s elapsed=23s eta=203s
[TRAIN] epoch=3 batch=24/26 loss=0.0000 lr=0.000096 speed=228.9s/s elapsed=25s eta=180s
[CKPT] epoch_end epoch=3 step=78
[EPOCH] 3/24: train_loss=1.7612 val_loss=1.6664 val_acc=0.2083 macro_f1=0.1710 * BEST
[TRAIN] epoch=4 batch=12/26 loss=0.0000 lr=0.000095 speed=231.1s/s el

In [16]:
with TELEMETRY.capture_cell_output("Cell 7: OOD Calibration"):
    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Once engine init hucresini calistirin.")

    adapter = STATE["adapter"]
    val_loader = STATE["loaders"].get("val")
    if val_loader is None or len(val_loader.dataset) == 0:
        raise RuntimeError("Validation loader bos; OOD kalibrasyonu yapilamaz.")

    calibration = adapter.calibrate_ood(val_loader)
    STATE["calibration"] = calibration

    num_classes = calibration.get("ood_calibration", {}).get("num_classes", 0)
    version = calibration.get("ood_calibration", {}).get("version", 0)
    print(f"[OOD] Kalibrasyon tamamlandi. classes={num_classes} version={version}")
    TELEMETRY.update_latest(
        {
            "phase": "ood_calibrated",
            "ood_num_classes": num_classes,
            "ood_version": version,
        }
    )

[OOD] Kalibrasyon tamamlandi. classes=6 version=1


In [17]:
with TELEMETRY.capture_cell_output("Cell 8: Adapter Save"):
    rt("Cell 8: adapter save started", phase="export")

    if STATE.get("adapter") is None:
        raise RuntimeError("Once Cell 5 calistirilmali.")

    crop_name = str(STATE.get("crop_name") or CROP_NAME).strip().lower()
    part_name = str(STATE.get("part_name") or PART_NAME or "unspecified").strip().lower() or "unspecified"
    checkpoint_dir = build_adapter_bundle_root(ROOT / "outputs" / "colab_notebook_training", crop_name, part_name)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    STATE["adapter"].save_adapter(str(checkpoint_dir))
    asset_dir = checkpoint_dir / "continual_sd_lora_adapter"
    STATE["adapter_export_dir"] = asset_dir

    print("Kaydedilen adapter klasoru:", asset_dir)
    if not asset_dir.exists():
        raise RuntimeError(f"Beklenen adapter klasoru bulunamadi: {asset_dir}")

    telemetry_adapter_root = TELEMETRY.artifacts_dir / "adapter_export" / crop_name / part_name / "continual_sd_lora_adapter"
    for path_in_adapter in sorted(asset_dir.rglob("*")):
        if path_in_adapter.is_file():
            relative_path = path_in_adapter.relative_to(LOCAL_OUTPUT_DIR).as_posix()
            TELEMETRY.copy_artifact_file(path_in_adapter, f"adapter_export/{relative_path}")
            print(" -", path_in_adapter.relative_to(ROOT))

    print("Telemetry adapter klasoru:", telemetry_adapter_root)
    TELEMETRY.update_latest(
        {
            "phase": "adapter_saved",
            "adapter_dir": str(telemetry_adapter_root),
        }
    )
    rt("Cell 8: adapter save completed", phase="export")


Cell 8: adapter save started
Kaydedilen adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/README.md
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_config.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_meta.json
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/adapter_model.safetensors
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/classifier.pth
 - outputs/colab_notebook_training/grape/leaf/continual_sd_lora_adapter/fusion.pth
Telemetry adapter klasoru: /content/bitirmeprojesi/outputs/colab_notebook_training/telemetry_runtime/telemetry/grape_leaf_2026-04-22_11-07-15/artifacts/adapter_export/grape/leaf/continual_sd_lora_adapter
Cell 8: adapter save completed


In [ ]:
with TELEMETRY.capture_cell_output("Cell 9: Final Evaluation"):
    import json
    from datetime import datetime, timezone
    from scripts.colab_notebook_helpers import (
        merge_training_summary_fields,
        notebook_artifact_root,
        persist_production_readiness_artifact,
        persist_validation_artifacts,
    )
    from src.adapter.independent_crop_adapter import IndependentCropAdapter
    from src.training.services.ood_benchmark import run_leave_one_class_out_benchmark
    from src.training.validation import evaluate_model_with_artifact_metrics

    if STATE.get("adapter") is None or STATE.get("loaders") is None:
        raise RuntimeError("Once engine init hucresi calistirilmali.")

    adapter = STATE["adapter"]
    loaders = STATE["loaders"]
    effective_params = dict(STATE.get("effective_params") or {})
    test_loader = loaders.get("test")
    if test_loader is None or len(test_loader.dataset) == 0:
        raise RuntimeError("Test loader bos. Final degerlendirme held-out test split ile yapilmali.")

    trainer = adapter._trainer
    if trainer is None:
        raise RuntimeError("Trainer hazir degil.")

    notebook_config = json.loads(json.dumps(BASE_CONFIG))
    notebook_config.setdefault("training", {})["continual"] = json.loads(json.dumps(STATE["continual_config"]))
    evaluation_cfg = notebook_config.get("training", {}).get("continual", {}).get("evaluation", {})
    artifact_root = notebook_artifact_root(ROOT)

    trainer.adapter_model.eval()
    trainer.classifier.eval()
    trainer.fusion.eval()

    classes = [name for name, _ in sorted(adapter.class_to_idx.items(), key=lambda item: item[1])]
    ood_loader = loaders.get("ood")

    def _evaluate_split(split_name: str, loader, *, artifact_subdir: str, label: str):
        evaluation = evaluate_model_with_artifact_metrics(trainer, loader, ood_loader=ood_loader)
        if evaluation is None:
            raise RuntimeError("Degerlendirme ornegi bulunamadi.")
        artifacts = persist_validation_artifacts(
            root=ROOT,
            y_true=evaluation.y_true,
            y_pred=evaluation.y_pred,
            classes=classes,
            telemetry=TELEMETRY,
            artifact_subdir=artifact_subdir,
            require_ood=bool(evaluation_cfg.get("require_ood_for_gate", True)),
            emit_metric_gate=bool(evaluation_cfg.get("emit_ood_gate", True)),
            ood_labels=evaluation.ood_labels,
            ood_scores=evaluation.ood_scores,
            ood_scores_by_method=evaluation.ood_scores_by_method,
            sure_ds_f1=evaluation.sure_ds_f1,
            conformal_empirical_coverage=evaluation.conformal_empirical_coverage,
            conformal_avg_set_size=evaluation.conformal_avg_set_size,
            ood_type_breakdown=evaluation.ood_type_breakdown,
            prediction_rows=evaluation.prediction_rows,
            context={
                "crop_name": CROP_NAME,
                "part_name": PART_NAME,
                "run_id": RUN_ID,
                "split_name": split_name,
                **evaluation.context,
            },
        )
        metrics = artifacts["metric_gate"]["metrics"]
        extras = []
        if metrics.get("ood_auroc") is not None:
            extras.append(f"ood_auroc={float(metrics['ood_auroc']):.4f}")
        if metrics.get("sure_ds_f1") is not None:
            extras.append(f"sure_ds_f1={float(metrics['sure_ds_f1']):.4f}")
        if metrics.get("conformal_empirical_coverage") is not None:
            extras.append(f"conformal_cov={float(metrics['conformal_empirical_coverage']):.4f}")
        suffix = " " + " ".join(extras) if extras else ""
        accuracy = float(artifacts["report_dict"].get("accuracy", 0.0))
        print(f"[{label}] ornek={len(evaluation.y_true)} sinif={len(classes)} accuracy={accuracy:.4f}{suffix}")
        return artifacts

    results = {}
    val_loader = loaders.get("val")
    if val_loader is not None and len(val_loader.dataset) > 0:
        results["validation"] = _evaluate_split(
            "val",
            val_loader,
            artifact_subdir="validation",
            label="DOGRULAMA (referans)",
        )

    results["test"] = _evaluate_split(
        "test",
        test_loader,
        artifact_subdir="test",
        label="TEST (ayrilmis)",
    )
    selected_split = "test" if "test" in results else "validation"
    selected_artifacts = results[selected_split]
    real_ood_present = ood_loader is not None and len(ood_loader.dataset) > 0
    ood_evidence_source = "real_ood_split" if real_ood_present else "unavailable"
    ood_evidence_metrics = dict(selected_artifacts["metric_gate"]["metrics"]) if real_ood_present else {}
    benchmark_summary = {}
    if (
        not real_ood_present
        and str(evaluation_cfg.get("ood_fallback_strategy", "held_out_benchmark")) == "held_out_benchmark"
        and bool(evaluation_cfg.get("ood_benchmark_auto_run", True))
    ):
        print("[OOD] Gercek OOD split bulunamadi; held-out benchmark fallback calisiyor...")
        benchmark_summary = run_leave_one_class_out_benchmark(
            crop_name=CROP_NAME,
            class_names=classes,
            loaders=loaders,
            config=notebook_config,
            device=DEVICE,
            artifact_root=artifact_root,
            adapter_factory=IndependentCropAdapter,
            run_id=RUN_ID,
            num_epochs=int(effective_params["EPOCHS"]),
            telemetry=TELEMETRY,
            emit_event=lambda event_type, payload: TELEMETRY.emit_event(event_type, payload, phase="evaluation"),
            min_classes=int(evaluation_cfg.get("ood_benchmark_min_classes", 3)),
        )
        ood_evidence_source = "held_out_benchmark"
        ood_evidence_metrics = dict(benchmark_summary.get("metrics", {}))

    readiness = persist_production_readiness_artifact(
        root=ROOT,
        classification_metric_gate=selected_artifacts.get("metric_gate"),
        classification_split=selected_split,
        ood_evidence_source=ood_evidence_source,
        ood_metrics=ood_evidence_metrics,
        context={
            "run_id": RUN_ID,
            "crop_name": CROP_NAME,
            "part_name": PART_NAME,
            "classification_split": selected_split,
            "ood_benchmark_status": benchmark_summary.get("status"),
            "ood_benchmark_passed": benchmark_summary.get("passed"),
        },
        require_ood=bool(evaluation_cfg.get("require_ood_for_gate", True)),
        telemetry=TELEMETRY,
    )

    STATE["evaluation_artifacts"] = results
    STATE["ood_benchmark"] = benchmark_summary
    STATE["production_readiness"] = readiness["payload"]
    plt.close("all")
    print(
        f"[OOD] kanit={readiness['payload'].get('ood_evidence_source', 'unavailable')} "
        f"durum={readiness['payload'].get('status', 'failed')} gecti={bool(readiness['payload'].get('passed', False))}"
    )

    TELEMETRY.update_latest(
        {
            "phase": "evaluation_complete",
            "evaluation_splits": sorted(results.keys()),
            "ood_evidence_source": readiness["payload"].get("ood_evidence_source"),
            "production_readiness": readiness["payload"].get("status"),
        }
    )
    print("[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.")

from scripts.colab_notebook_helpers import build_notebook_completion_report, maybe_auto_disconnect_colab_runtime, merge_training_summary_fields

REPO_RUN_EXPORTS = save_run_outputs_to_repo()
notebook_export_result = export_current_colab_notebook(REPO_NOTEBOOK_OUTPUT_PATH)

extra_entries = [
    {
        "path": build_adapter_bundle_root(REPO_OUTPUT_DIR, CROP_NAME, PART_NAME) / "continual_sd_lora_adapter",
        "category": "adapter_export",
        "priority": "high",
        "title_tr": "Repo mirror adapter export klasoru",
        "description_tr": "Repo mirror icindeki adapter export klasoru.",
        "reader_goal": "Export edilen adapter klasorunu bulmak",
        "generated_by": "notebook_2",
        "decision_importance": "deploy_handoff",
        "read_order": 70,
    },
    {
        "path": REPO_NOTEBOOK_OUTPUT_PATH,
        "category": "adapter_export",
        "priority": "medium",
        "title_tr": "Calistirilmis notebook exportu",
        "description_tr": "Bu kosuda calisan notebook'un kaydedilmis kopyasi.",
        "reader_goal": "Notebook'u ayni ciktiyla tekrar incelemek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 71,
    },
    {
        "path": REPO_TELEMETRY_DIR / "events.jsonl",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Telemetry event logu",
        "description_tr": "Notebook olayi bazli telemetry kaydi.",
        "reader_goal": "Notebook akisini olay bazinda incelemek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 80,
    },
    {
        "path": REPO_TELEMETRY_DIR / "runtime.log",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Runtime logu",
        "description_tr": "Notebook runtime boyunca yazilan metin logu.",
        "reader_goal": "Calisma sirasindaki log ciktilarini okumak",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 81,
    },
    {
        "path": REPO_TELEMETRY_DIR / "latest_status.json",
        "category": "logs_and_checkpoints",
        "priority": "low",
        "title_tr": "Son durum ozeti",
        "description_tr": "Notebook'un son durum snapshot'i.",
        "reader_goal": "Kosunun son durumunu hizli kontrol etmek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 82,
    },
    {
        "path": REPO_TELEMETRY_DIR / "summary.json",
        "category": "logs_and_checkpoints",
        "priority": "high",
        "title_tr": "Telemetry ozeti",
        "description_tr": "Notebook final ozet dosyasi.",
        "reader_goal": "Notebook final ozetini okumak",
        "generated_by": "notebook_2",
        "decision_importance": "run_overview",
        "read_order": 83,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "best_checkpoint.json",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Best checkpoint manifesti",
        "description_tr": "En iyi checkpoint'in repo mirror manifesti.",
        "reader_goal": "Hangi checkpoint secildigini gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 84,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "latest_checkpoint.json",
        "category": "logs_and_checkpoints",
        "priority": "medium",
        "title_tr": "Latest checkpoint manifesti",
        "description_tr": "Son checkpoint manifesti.",
        "reader_goal": "Checkpoint akisini gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 85,
    },
    {
        "path": REPO_CHECKPOINT_STATE_DIR / "checkpoint_index.json",
        "category": "logs_and_checkpoints",
        "priority": "low",
        "title_tr": "Checkpoint indexi",
        "description_tr": "Mirror edilen checkpoint manifest listesi.",
        "reader_goal": "Checkpoint kayitlarini toplu gormek",
        "generated_by": "notebook_2",
        "decision_importance": "runtime_diagnostic",
        "read_order": 86,
    },
]
summary_payload = merge_training_summary_fields(
    root=ROOT,
    telemetry=TELEMETRY,
    payload={
        "recommendation": {
            "decision": str(STATE.get("recommendation_decision") or "pending"),
            "change_count": int((STATE.get("recommendation_report") or {}).get("change_count", 0)),
            "manual_overrides": dict(MANUAL_PARAM_OVERRIDES or {}),
        },
        "run_id": RUN_ID,
        "run_label": RUN_ID,
        "crop_name": CROP_NAME,
        "part_name": PART_NAME,
        "notebook_surface": "2_interactive_adapter_training.ipynb",
        "dataset_roots": {
            "runtime_dataset_root": RUNTIME_DATASET_ROOT,
            "runtime_dataset_key": str(STATE.get("runtime_dataset_key") or ""),
            "selected_dataset_name": str(STATE.get("selected_dataset_name") or ""),
            "selected_runtime_dataset_root": str(STATE.get("selected_dataset_root") or ""),
            "resolved_ood_root": str(STATE.get("resolved_ood_root") or ""),
            "resolved_runtime_dataset_root": str(STATE.get("runtime_dataset_root") or ""),
        },
        "notebook_parameters": {
            "epochs": (STATE.get("effective_params") or {}).get("EPOCHS"),
            "batch_size": (STATE.get("effective_params") or {}).get("BATCH_SIZE"),
            "learning_rate": (STATE.get("effective_params") or {}).get("LEARNING_RATE"),
            "lora_r": (STATE.get("effective_params") or {}).get("LORA_R"),
            "ood_factor": (STATE.get("effective_params") or {}).get("OOD_FACTOR"),
            "mixed_precision": (STATE.get("effective_params") or {}).get("MIXED_PRECISION"),
            "num_workers": (STATE.get("effective_params") or {}).get("NUM_WORKERS"),
            "checkpoint_every_n_steps": (STATE.get("effective_params") or {}).get("CHECKPOINT_EVERY_N_STEPS"),
        },
        "export_paths": {
            "repo_run_dir": str(REPO_RUN_DIR),
            "repo_output_dir": str(REPO_OUTPUT_DIR),
            "repo_telemetry_dir": str(REPO_TELEMETRY_DIR),
            "repo_checkpoint_state_dir": str(REPO_CHECKPOINT_STATE_DIR),
            "executed_notebook_path": str(notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH),
            "adapter_export_dir": str(STATE.get("adapter_export_dir") or ""),
        },
        "access_check": STATE.get("access_report", {}),
        "readiness_summary": {
            "status": (STATE.get("production_readiness") or {}).get("status"),
            "passed": (STATE.get("production_readiness") or {}).get("passed"),
            "ood_evidence_source": (STATE.get("production_readiness") or {}).get("ood_evidence_source"),
        },
        "created_at": datetime.now(timezone.utc).isoformat(),
    },
    extra_entries=extra_entries,
)
TELEMETRY.merge_summary_metadata(
    {
        "access_check": STATE.get("access_report", {}),
        "repo_paths": {
            "repo_run_dir": str(REPO_RUN_DIR),
            "repo_output_dir": str(REPO_OUTPUT_DIR),
            "repo_telemetry_dir": str(REPO_TELEMETRY_DIR),
            "repo_checkpoint_state_dir": str(REPO_CHECKPOINT_STATE_DIR),
        },
        "training_summary": summary_payload,
    }
)
TELEMETRY.close(
    {
        "status": "ok",
        "evaluation_splits": sorted((STATE.get("evaluation_artifacts") or {}).keys()),
        "cell_outputs_dir": str(TELEMETRY.artifacts_dir / "cell_outputs"),
        "repo_run_dir": str(REPO_RUN_DIR),
        "run_label": RUN_ID,
    }
)
REPO_RUN_EXPORTS = save_run_outputs_to_repo()
for key in sorted(REPO_RUN_EXPORTS):
    print(f"[RUNS] {key} -> {REPO_RUN_EXPORTS[key]}")
print(f"[RUNS] notebook -> {REPO_NOTEBOOK_OUTPUT_PATH}")
effective_params = dict(STATE.get("effective_params") or {})
if bool(effective_params.get("AUTO_PUSH_TO_GITHUB", AUTO_PUSH_TO_GITHUB)):
    try:
        git_push_report = push_repo_run_to_github(
            ROOT,
            RUN_ID,
            run_relative_dir=REPO_RUN_DIR.relative_to(ROOT),
            remote_name=effective_params.get("AUTO_PUSH_REMOTE_NAME", AUTO_PUSH_REMOTE_NAME),
            branch=effective_params.get("AUTO_PUSH_BRANCH", AUTO_PUSH_BRANCH),
            print_fn=print,
        )
    except RuntimeError as exc:
        print(f"[GIT] Auto-push skipped: {exc}")
        git_push_report = {"enabled": True, "pushed": False, "run_dir": str(REPO_RUN_DIR), "error": str(exc)}
else:
    git_push_report = {"enabled": False, "pushed": False, "run_dir": str(REPO_RUN_DIR)}
STATE["git_push_report"] = git_push_report
disconnect_report = build_notebook_completion_report(
    state=STATE,
    telemetry=TELEMETRY,
    repo_run_exports=REPO_RUN_EXPORTS,
    notebook_export_path=notebook_export_result or REPO_NOTEBOOK_OUTPUT_PATH,
)
STATE["auto_disconnect_report"] = disconnect_report
print(f"[COLAB] completion checks -> {disconnect_report['checks']}")
maybe_auto_disconnect_colab_runtime(
    enabled=bool(effective_params.get("AUTO_DISCONNECT_RUNTIME", AUTO_DISCONNECT_RUNTIME)),
    grace_period_sec=int(effective_params.get("AUTO_DISCONNECT_GRACE_SECONDS", AUTO_DISCONNECT_GRACE_SECONDS)),
    telemetry=TELEMETRY,
    completion_report=disconnect_report,
)


[DOGRULAMA (referans)] ornek=216 sinif=6 accuracy=0.8981 ood_auroc=0.8439 sure_ds_f1=0.4320 conformal_cov=0.9583
[TEST (ayrilmis)] ornek=216 sinif=6 accuracy=0.8889 ood_auroc=0.8296 sure_ds_f1=0.4441 conformal_cov=0.9630
[OOD] kanit=real_ood_split durum=failed gecti=False
[DONE] Dogrulama ve held-out test artefaktlari kaydedildi.
